## Cleaned Dataset Quality Checks

In [4]:
import pandas as pd
cleaned_path = "../data/cleaned/blockbuster_movies_cleaned_starter.csv"

movies = pd.read_csv(cleaned_path)

movies.head()

,movie_id,title,release_date,release_year,release_month,studio,genre,ip_type,franchise,production_budget,worldwide_gross,runtime,audience_score,vote_count,popularity
0,262500,Insurgent,2015-03-18,2015,3,"Summit Entertainment, Red Wagon Entertainment,...","Action, Science Fiction, Thriller",NaN,NaN,110000000,297002527,119,6.359,10451,7.2938
1,131634,The Hunger Games: Mockingjay - Part 2,2015-11-18,2015,11,"Lionsgate, Color Force, Studio Babelsberg","Action, Adventure, Science Fiction",NaN,NaN,160000000,653428261,137,6.900,13356,7.5180
2,135397,Jurassic World,2015-06-06,2015,6,"Amblin Entertainment, Universal Pictures, Lege...","Adventure, Science Fiction, Thriller",NaN,NaN,150000000,1671537444,124,6.701,21608,13.0151
3,266647,Pan,2015-09-24,2015,9,"Moving Picture Company, Berlanti Productions, ...","Fantasy, Adventure, Family, Action",NaN,NaN,150000000,128388320,111,6.000,2852,3.6698
4,140607,Star Wars: The Force Awakens,2015-12-15,2015,12,"Lucasfilm Ltd., Bad Robot","Adventure, Action, Science Fiction",NaN,NaN,245000000,2068223624,136,7.249,20570,10.9203


In [5]:
movies.shape

(293, 15)

In [6]:
movies.columns

Index(['movie_id', 'title', 'release_date', 'release_year', 'release_month',
       'studio', 'genre', 'ip_type', 'franchise', 'production_budget',
       'worldwide_gross', 'runtime', 'audience_score', 'vote_count',
       'popularity'],
      dtype='object')

In [7]:
movies[["title", "ip_type", "franchise"]].head(10)

,title,ip_type,franchise
0,Insurgent,NaN,NaN
1,The Hunger Games: Mockingjay - Part 2,NaN,NaN
2,Jurassic World,NaN,NaN
3,Pan,NaN,NaN
4,Star Wars: The Force Awakens,NaN,NaN
5,Inside Out,NaN,NaN
6,The Revenant,NaN,NaN
7,The Martian,NaN,NaN
8,Tomorrowland,NaN,NaN
9,Fantastic Four,NaN,NaN


In [8]:
movies.isna().sum()

movie_id               0
title                  0
release_date           0
release_year           0
release_month          0
studio                 8
genre                  1
ip_type              293
franchise            293
production_budget      0
worldwide_gross        0
runtime                0
audience_score         0
vote_count             0
popularity             0
dtype: int64

In [9]:
movies["movie_id"].duplicated().sum()

np.int64(0)

In [10]:
zero_budget = movies[movies["production_budget"] == 0]

zero_budget[["title", "release_year", "production_budget", "worldwide_gross"]]

,title,release_year,production_budget,worldwide_gross


In [11]:
zero_revenue = movies[movies["worldwide_gross"] == 0]

zero_revenue[["title", "release_year", "production_budget", "worldwide_gross"]]

,title,release_year,production_budget,worldwide_gross
88,Mowgli: Legend of the Jungle,2018,175000000,0
112,Triple Frontier,2019,115000000,0
134,"Enthronement Ceremony - October 22, 2019",2019,112105610,0
137,6 Underground,2019,150000000,0
146,Artemis Fowl,2020,125000000,0
159,Angry Birds Plush: Egg Trilogy,2021,200000000,0
182,The Adam Project,2022,116000000,0
194,Day Shift,2022,100000000,0
232,Beverly Hills Cop: Axel F,2024,150000000,0
233,Habitat,2024,100000000,0


In [12]:
movies["production_budget"] = movies["production_budget"].replace(0, pd.NA)
movies["worldwide_gross"] = movies["worldwide_gross"].replace(0, pd.NA)

In [13]:
movies[["production_budget", "worldwide_gross"]].isna().sum()

production_budget     0
worldwide_gross      19
dtype: int64

In [14]:
movies["has_financial_data"] = (
    movies["production_budget"].notna()
    & movies["worldwide_gross"].notna()
)

In [15]:
movies["has_financial_data"].value_counts()

has_financial_data
True     274
False     19
Name: count, dtype: int64

In [17]:
movies["estimated_break_even"] = movies["production_budget"] * 2.5

movies["profitability_ratio"] = (
    movies["worldwide_gross"] / movies["estimated_break_even"]
)
#estimated_break_even uses your project assumption that a blockbuster needs about 2.5x its production budget to cover production plus marketing/distribution costs.
# profitability_ratio compares worldwide gross to that estimated break-even number:
# Above 1.0 means likely profitable by this rough rule.
# Below 1.0 means likely under break-even.
# Missing revenue gives a missing ratio.

In [21]:
def classify_hit_status(row):
    if pd.isna(row["profitability_ratio"]):
        return "Unknown"

    if row["profitability_ratio"] >= 1.6:
        return "Major Hit"

    if row["profitability_ratio"] >= 1.2:
        return "Hit"

    if row["profitability_ratio"] >= 1.0:
        return "Break Even"

    return "Miss"

In [22]:
movies["hit_status"] = movies.apply(classify_hit_status, axis=1)

movies["hit_status"].value_counts()
# Major Hit  = 4x+ production budget
# Hit        = 3x-4x production budget
# Break Even = 2.5x-3x production budget
# Miss       = below estimated break-even
# Unknown    = missing worldwide gross

hit_status
Miss          144
Major Hit      72
Hit            42
Unknown        19
Break Even     16
Name: count, dtype: int64

In [23]:
def classify_budget_tier(row):
    if pd.isna(row["production_budget"]):
        return "Unknown"

    if row["production_budget"] >= 250_000_000:
        return "Mega Budget"

    if row["production_budget"] >= 150_000_000:
        return "High Budget"

    return "Standard Blockbuster"

In [24]:
movies["budget_tier"] = movies.apply(classify_budget_tier, axis=1)

movies["budget_tier"].value_counts()

budget_tier
High Budget             146
Standard Blockbuster    117
Mega Budget              30
Name: count, dtype: int64

In [25]:
def classify_release_era(row):
    if row["release_year"] <= 2019:
        return "Pre-COVID"

    if row["release_year"] <= 2021:
        return "COVID Era"

    return "Post-COVID"

In [26]:
movies["release_era"] = movies.apply(classify_release_era, axis=1)

movies["release_era"].value_counts()

release_era
Pre-COVID     141
Post-COVID    116
COVID Era      36
Name: count, dtype: int64

In [27]:
analysis_ready_path = "../data/cleaned/blockbuster_movies_analysis_ready.csv"

movies.to_csv(analysis_ready_path, index=False)

print("Saved:", analysis_ready_path)
print(movies.shape)

Saved: ../data/cleaned/blockbuster_movies_analysis_ready.csv
(293, 21)


In [28]:
movies["hit_status"].value_counts()

hit_status
Miss          144
Major Hit      72
Hit            42
Unknown        19
Break Even     16
Name: count, dtype: int64

In [29]:
movies.groupby("release_era")["hit_status"].value_counts()

release_era  hit_status
COVID Era    Miss          26
             Major Hit      4
             Hit            3
             Unknown        2
             Break Even     1
Post-COVID   Miss          62
             Major Hit     23
             Hit           16
             Unknown       13
             Break Even     2
Pre-COVID    Miss          56
             Major Hit     45
             Hit           23
             Break Even    13
             Unknown        4
Name: count, dtype: int64

In [30]:
movies.groupby("budget_tier")["hit_status"].value_counts()

budget_tier           hit_status
High Budget           Miss          69
                      Major Hit     41
                      Hit           19
                      Unknown        9
                      Break Even     8
Mega Budget           Miss          15
                      Major Hit      8
                      Hit            6
                      Unknown        1
Standard Blockbuster  Miss          60
                      Major Hit     23
                      Hit           17
                      Unknown        9
                      Break Even     8
Name: count, dtype: int64

In [31]:
movies.groupby("budget_tier")["profitability_ratio"].mean().sort_values()

budget_tier
Standard Blockbuster    1.066664
Mega Budget             1.192128
High Budget             1.283317
Name: profitability_ratio, dtype: object